In [ ]:
from collections import Counter
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if not (repo_root / "src" / "data").is_dir():
    raise FileNotFoundError("Open the notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(repo_root / "src"))

from data.datasets import get_dataset
from data.splits import get_train_val_split

dataset_name = "Oxford-IIIT Pet classification"
data_dir = repo_root / "data" / "raw" / "iit_pet"

# Oxford-IIIT Pet provides official trainval and test partitions.
trainval_dataset = get_dataset(
    split="trainval",
    dataset=dataset_name,
    data_dir=data_dir,
    download=True,
)
test_dataset = get_dataset(
    split="test",
    dataset=dataset_name,
    data_dir=data_dir,
)

# Stratify on breed labels so each class is represented in both subsets.
# Torchvision stores the Oxford-IIIT Pet category IDs in _labels.
train_dataset, val_dataset = get_train_val_split(
    trainval_dataset,
    ratio=0.2,
    seed=42,
    labels=trainval_dataset._labels,
)

assert len(train_dataset) + len(val_dataset) == len(trainval_dataset)
assert set(train_dataset.indices).isdisjoint(val_dataset.indices)

for name, subset in (("Train", train_dataset), ("Validation", val_dataset)):
    counts = Counter(trainval_dataset._labels[i] for i in subset.indices)
    print(
        f"{name}: {len(subset)} images, {len(counts)} classes "
        f"({min(counts.values())}-{max(counts.values())} per class)"
    )

print(f"Test: {len(test_dataset)} images (official held-out set)")
image, label = train_dataset[0]
print(f"Example: image size={image.size}, label={label}")
